In [ ]:
import evaluate
from transformers import BartTokenizer, BartForConditionalGeneration

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

tokenizer = BartTokenizer.from_pretrained("./bart-mtg-model")
model = BartForConditionalGeneration.from_pretrained("./bart-mtg-model")


test_inputs = [
    "What does Sol Ring do?",
    "What does Lightning Bolt do??",
    "What is in the Counter Blitz Collector's Edition (FINAL FANTASY X) deck?"
]

test_references = [
    "Sol Ring is a Artifact. {T}: Add {C}{C}.",
    "Lightning Bolt is a Instant. Lightning Bolt deals 3 damage to any target.",
    "The Counter Blitz Collector's Edition (FINAL FANTASY X) deck contains cards such as Altered Ego, An Offer You Can't Refuse, Arcane Signet, Ash Barrens, Auron, Bane of Progress, Blitzball Stadium, Bred for the Hunt, Brushland, Canopy Vista."
]

predictions = []


for text in test_inputs:
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model.generate(inputs["input_ids"], max_length=64)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred)

bleu_score = bleu.compute(
    predictions=predictions,
    references=[[r] for r in test_references]
)

rouge_score = rouge.compute(
    predictions=predictions,
    references=test_references
)

print("BLEU:", bleu_score["bleu"])
print("ROUGE-1:", rouge_score["rouge1"])
print("ROUGE-L:", rouge_score["rougeL"])


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


BLEU: 0.4554461778345942
ROUGE-1: 0.7878787878787877
ROUGE-L: 0.7878787878787877


## Score Explanation

The BLEU score came in at 0.45 which is a remarkable score for the chatbot. BLEU was picked as the primary scoring since it measures overlap of N-grams. A not perfect score demonstrates the model is not just regurgitating the information stored in the dataset but attempting to provide accurate, differemt summaries using different words. ROUGE was added in for information recall and came in at 0.78. ROUGE was included not only to highlight the difference between structure and information recall, but because for an information-heavy corpus, accurate summaries are the main measure; If the model matches the structure of input but summaries are wrong, then a referential bot would be useless.